# 02 · Cessation pharmacotherapy receipt

Varenicline and NRT prescriptions are identified from EHR drug exposures (RxNorm). Each person is classified
as *Varenicline only*, *NRT only*, *Both* (combination therapy), or *No treatment*.

## 1. Drug exposures (drugs used in nicotine dependence)

In [ ]:
library(tidyverse)
library(bigrquery)

# This query represents dataset "Test1" for domain "drug" and was generated for All of Us Controlled Tier Dataset v8
dataset_84952154_drug_sql <- paste("
    SELECT
        d_exposure.person_id,
        d_exposure.drug_concept_id,
        d_standard_concept.concept_name as standard_concept_name,
        d_standard_concept.concept_code as standard_concept_code,
        d_standard_concept.vocabulary_id as standard_vocabulary,
        d_exposure.drug_exposure_start_datetime,
        d_exposure.drug_exposure_end_datetime,
        d_exposure.verbatim_end_date,
        d_exposure.drug_type_concept_id,
        d_type.concept_name as drug_type_concept_name,
        d_exposure.stop_reason,
        d_exposure.refills,
        d_exposure.quantity,
        d_exposure.days_supply,
        d_exposure.sig,
        d_exposure.route_concept_id,
        d_route.concept_name as route_concept_name,
        d_exposure.lot_number,
        d_exposure.visit_occurrence_id,
        d_visit.concept_name as visit_occurrence_concept_name,
        d_exposure.drug_source_value,
        d_exposure.drug_source_concept_id,
        d_source_concept.concept_name as source_concept_name,
        d_source_concept.concept_code as source_concept_code,
        d_source_concept.vocabulary_id as source_vocabulary,
        d_exposure.route_source_value,
        d_exposure.dose_unit_source_value 
    FROM
        ( SELECT
            * 
        FROM
            `drug_exposure` d_exposure 
        WHERE
            (
                drug_concept_id IN (SELECT
                    DISTINCT ca.descendant_id 
                FROM
                    `cb_criteria_ancestor` ca 
                JOIN
                    (SELECT
                        DISTINCT c.concept_id       
                    FROM
                        `cb_criteria` c       
                    JOIN
                        (SELECT
                            CAST(cr.id as string) AS id             
                        FROM
                            `cb_criteria` cr             
                        WHERE
                            concept_id IN (21604817)             
                            AND full_text LIKE '%_rank1]%'       ) a 
                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                            OR c.path LIKE CONCAT('%.', a.id) 
                            OR c.path LIKE CONCAT(a.id, '.%') 
                            OR c.path = a.id) 
                    WHERE
                        is_standard = 1 
                        AND is_selectable = 1) b 
                        ON (ca.ancestor_id = b.concept_id)))) d_exposure 
        LEFT JOIN
            `concept` d_standard_concept 
                ON d_exposure.drug_concept_id = d_standard_concept.concept_id 
        LEFT JOIN
            `concept` d_type 
                ON d_exposure.drug_type_concept_id = d_type.concept_id 
        LEFT JOIN
            `concept` d_route 
                ON d_exposure.route_concept_id = d_route.concept_id 
        LEFT JOIN
            `visit_occurrence` v 
                ON d_exposure.visit_occurrence_id = v.visit_occurrence_id 
        LEFT JOIN
            `concept` d_visit 
                ON v.visit_concept_id = d_visit.concept_id 
        LEFT JOIN
            `concept` d_source_concept 
                ON d_exposure.drug_source_concept_id = d_source_concept.concept_id", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
drug_84952154_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "drug_84952154",
  "drug_84952154_*.csv")
message(str_glue('The data will be written to {drug_84952154_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_84952154_drug_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  drug_84952154_path,
  destination_format = "CSV")

# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {drug_84952154_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(standard_concept_name = col_character(), standard_concept_code = col_character(), standard_vocabulary = col_character(), drug_type_concept_name = col_character(), stop_reason = col_character(), sig = col_character(), route_concept_name = col_character(), lot_number = col_character(), visit_occurrence_concept_name = col_character(), drug_source_value = col_character(), source_concept_name = col_character(), source_concept_code = col_character(), source_vocabulary = col_character(), route_source_value = col_character(), dose_unit_source_value = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
nicotine_treatment_df <- read_bq_export_from_workspace_bucket(drug_84952154_path)

In [ ]:
# Drug concepts for Supplementary Table S3
nicotine_treatment_df %>%
  distinct(standard_concept_name, standard_concept_code, standard_vocabulary) %>%
  arrange(standard_vocabulary, standard_concept_name) %>%
  write_csv("supp_table_S3_drug_concepts.csv")

## 2. Classify drug type and summarise per person

In [ ]:
treatment_categories <- nicotine_treatment_df %>%
  mutate(
    # Convert datetime strings to POSIXct objects
    drug_exposure_start_datetime = as.POSIXct(drug_exposure_start_datetime, format = "%Y-%m-%d %H:%M:%S", tz = "UTC"),
    drug_exposure_end_datetime = as.POSIXct(drug_exposure_end_datetime, format = "%Y-%m-%d %H:%M:%S", tz = "UTC"),
    
    # Create drug type categories
    drug_type = case_when(
      str_detect(standard_concept_name, "varenicline") ~ "Varenicline",
      str_detect(standard_concept_name, "nicotine") ~ "NRT",
      TRUE ~ "Other"
    ),
    
    # Calculate treatment duration in days
    treatment_duration_days = as.numeric(difftime(drug_exposure_end_datetime, 
                                                 drug_exposure_start_datetime, 
                                                 units = "days"))
  ) %>%
  # Sort by person_id and start datetime to prepare for gap calculations
  arrange(person_id, drug_exposure_start_datetime) %>%
  group_by(person_id) %>%
  mutate(
    # Calculate time to next episode (gap between current end and next start)
    next_episode_start = lead(drug_exposure_start_datetime),
    time_to_next_episode_days = as.numeric(difftime(next_episode_start, 
                                                   drug_exposure_end_datetime, 
                                                   units = "days")),
    
    # Add episode number for each patient
    episode_number = row_number()
  ) %>%
  ungroup()

In [ ]:
treatment_categories <- treatment_categories %>%
  mutate(
    drug_exposure_start_date = as.Date(drug_exposure_start_datetime),
    drug_exposure_end_date = as.Date(drug_exposure_end_datetime)
  )

In [ ]:
patient_exposure <- treatment_categories %>%
  group_by(person_id) %>%
  summarise(
    has_nrt = any(drug_type == 'NRT', na.rm = TRUE),
    has_varenicline = any(drug_type == 'Varenicline', na.rm = TRUE),
    total_treatments = n_distinct(standard_concept_code),
    first_treatment_date = min(drug_exposure_start_date, na.rm = TRUE),
    .groups = 'drop'
  ) %>%
  left_join(
    treatment_categories %>%
      group_by(person_id) %>%
      summarise(
        effective_end_dates = list(
          if_else(is.na(drug_exposure_end_date), 
                 drug_exposure_start_date, 
                 drug_exposure_end_date)
        ),
        .groups = 'drop'
      ) %>%
      rowwise() %>%
      mutate(
        last_treatment_date = max(unlist(effective_end_dates), na.rm = TRUE)
      ) %>%
      select(person_id, last_treatment_date),
    by = "person_id"
  ) %>%
  mutate(
    treatment_group = case_when(
      has_nrt & has_varenicline ~ 'Both NRT + Varenicline',
      has_nrt & !has_varenicline ~ 'NRT Only',
      !has_nrt & has_varenicline ~ 'Varenicline Only',
      TRUE ~ 'Neither'
    ),
    treatment_duration_days = as.numeric(last_treatment_date - first_treatment_date) + 1
  )

table(patient_exposure$treatment_group)

## 3. Attach treatment to the smoking cohort

In [ ]:
final_smokers_cohort_filtered <- load_from_bucket("smokers_cohort.csv")

# NOTE (cleanup): the original notebook only added `has_treatment` here; the
# treatment-group columns used later came from a previously saved CSV. They
# are joined explicitly below.
DF_smoker <- final_smokers_cohort_filtered %>%
  mutate(has_treatment = if_else(person_id %in% patient_exposure$person_id, 1, 0)) %>%
  left_join(patient_exposure %>%
              select(person_id, has_nrt, has_varenicline, treatment_group,
                     first_treatment_date, last_treatment_date),
            by = "person_id")

table(DF_smoker$has_treatment)
table(DF_smoker$treatment_group, useNA = "ifany")

In [ ]:
DF_smoker <- DF_smoker %>%
  mutate(
    # Numeric treatment group coding
    treatment_group_coded = case_when(
      is.na(treatment_group) ~ 0,  # No treatment
      treatment_group == "NRT Only" ~ 1,
      treatment_group == "Varenicline Only" ~ 2,
      treatment_group == "Both NRT + Varenicline" ~ 3
    ),
    # Factor version for modeling
    treatment_group_factor = factor(treatment_group_coded,
                                    levels = 0:3,
                                    labels = c("No Treatment", "NRT",
                                               "Varenicline", "Both"))
  )

# NOTE (cleanup): people whose only drug exposure is "Other" get treatment_group
# "Neither": has_treatment = 1 but treatment_group_factor = NA, so they drop out
# of the Aim 2 models. Check that this matches what the paper describes.
save_to_bucket(DF_smoker, "cohort_treatment.csv")